# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (provisional — can change until end of Week 4).

Why this one: it's the lane with the most existing scaffolding to learn from — the starter pipeline in this repo builds it end to end (baseline score → model → ranked queue with reason codes), and its own verified results already show a learned ranking clearly beats a fixed rule on this data (baseline Precision@50 = 0.240 vs. random forest Precision@50 = 0.740, from `outputs/model_report.md`). That's evidence the pattern here is real but too tangled for a hand-written rule alone — exactly the condition where ML earns its place instead of just a dashboard.

The numbers in Section 3 back this further: a large share of the starter slice shows a declining trend *with* real demand behind it, and a majority of visible pages sit below their position tier's expected CTR — so there's a genuinely large candidate pool to rank, not a rare-event needle-in-a-haystack problem.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Question:** Which content pages should a reviewer look at first for a refresh (rewrite, expand, protect, prune, or monitor)?

**Decision it improves:** how a content editor with limited weekly review capacity allocates their attention across a large page inventory — not "predict decline" in the abstract, but "which N pages go on this week's review list."

**Who acts, and what they do:** an SEO/content editor (or a small content team) opens the top of a ranked review queue, reads the reason codes attached to each page (e.g. "declining with demand," "low CTR for its position," "stale but still visible"), and decides on an action per page — rewrite, expand, protect, prune, or just keep monitoring.

**Cost of a wrong call:**
- *False positive* (a page is ranked high but didn't actually need attention): wastes scarce editor hours that could have gone to a page that really was declining.
- *False negative* (a genuinely declining, high-value page never surfaces in the queue): the page keeps losing visibility/clicks silently until someone notices by chance — a real, compounding traffic/lead cost.
- Because editor time is the scarce resource, **ranking quality (precision@K)** matters more than raw classification accuracy — being right about the *top 50* matters more than being right on average across all 30,000 pages.

**Why data or ML helps at all:** a simple rule-based baseline already exists (documented in the lane guide, with reason codes like `stale_visible_page` and `declining_with_demand`), so a plain if-statement approach isn't unavailable — it's just weak. The guide's own measured comparison shows a learned model roughly triples the baseline's Precision@50 on this data. That gap is the actual justification for ML here: the signal is real, but it's spread across many tangled, interacting columns (trend, position, CTR, freshness, volume, content type) in a way a hand-written rule captures only partially.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV and checking how big the candidate pool for "needs review" actually is — using the same reason-code logic the lane guide documents for the starter baseline (`stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`), computed fresh here rather than just cited from the guide.

Note up front: `trend_direction` / `trend_pct` define the starter's proxy label (`is_declining_label`) — shown here only as a descriptive number, never as a model feature later.

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_clients = df["client_id"].nunique()
print(f"Starter slice: {n_rows:,} rows across {n_clients} pseudonymized clients")

# 1) How much of the slice already shows a declining trend WITH real demand behind it
#    (the lane guide's own reason code: declining_with_demand)
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
pct_declining_demand = len(declining_with_demand) / n_rows * 100
print(f"declining_with_demand (trend=down & impressions_90d>=100): "
      f"{len(declining_with_demand):,} rows ({pct_declining_demand:.1f}% of the slice)")

# 2) Of the visible pages (enough impressions to matter), how many look CTR-weak
#    for their position tier -- reason code: low_ctr_visible_page
visible = df[df["impressions_90d"] >= 500]
low_ctr_visible = visible[
    (visible["avg_position"] > 0) & (visible["avg_position"] <= 20) & (visible["ctr"] < 0.5)
]
pct_low_ctr = len(low_ctr_visible) / len(visible) * 100
print(f"Visible pages (impressions_90d>=500): {len(visible):,}")
print(f"  of which low_ctr_visible_page (position 1-20, ctr<0.5%): "
      f"{len(low_ctr_visible):,} ({pct_low_ctr:.1f}% of visible pages)")

# 3) Base rate of the starter proxy label itself -- descriptive only, this column
#    is NEVER used as a model feature (it's what defines the label)
down_rate = (df["trend_direction"] == "down").mean() * 100
print(f"trend_direction == 'down' base rate: {down_rate:.1f}% of all {n_rows:,} rows")


Starter slice: 30,000 rows across 32 pseudonymized clients
declining_with_demand (trend=down & impressions_90d>=100): 13,152 rows (43.8% of the slice)
Visible pages (impressions_90d>=500): 16,726
  of which low_ctr_visible_page (position 1-20, ctr<0.5%): 9,759 (58.3% of visible pages)
trend_direction == 'down' base rate: 54.2% of all 30,000 rows


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- *Observed*: which pages, in this snapshot, show a declining trend, weak CTR for their position, or staleness alongside real demand.
- *Directional / associative*: which signals tend to travel together with those patterns (e.g. word count, freshness, content type) — correlation, not causation.
- *Decision-support*: a ranked review queue with reason codes, meant to help a human editor spend limited time on the most promising candidates first — not a guarantee about any individual page.

**What it will never claim:**
- That refreshing a page *caused* a recovery — that needs an experiment or another causal design, which this data alone can't provide (Section 6/7 of the lane guide is explicit about this).
- Anything about Google's actual ranking algorithm — the data shows observed search/engagement outcomes, not the algorithm's internals.
- Anything about "AI rankings" or "AI citations" — the only AI-related signal here is `ai_sessions_90d`, which measures click-throughs from AI tools, not whether or how an AI assistant referenced the page.

**A base-rate caution worth flagging now:** `trend_direction == "down"` is unusually common in this slice (Section 3 shows the exact number) — high enough that a naive "always predict down" rule already scores well on plain accuracy. That means accuracy alone would be a misleading way to judge any later model; Precision@K on the ranked queue is the metric that actually matches how this output gets used.

**One leakage rule I'm committing to now, before any modeling:** `trend_direction` and `trend_pct` define the starter's proxy label — they get cited descriptively (as above) but are never allowed into the feature set later. Using them as inputs would just teach a model to repeat its own label back.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.